In [10]:
import pandas as pd


# load your final processed CSV
df = pd.read_csv("../Preprocessing_2/out/preprocessed_xy1_crop224_target_class/processed_metadata_all_samples.csv")


In [11]:
df['sample_type'].value_counts()

sample_type
pos         1234
neg          912
hard_neg     881
Name: count, dtype: int64

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# load your final processed CSV
df = pd.read_csv("../Preprocessing_2/out/preprocessed_xy1_crop224_target_class/processed_metadata_all_samples.csv")

# keep only successfully processed rows
df = df[df["processing_status"] == "ok"].copy()

In [13]:
# ---------------------------
# Step 1: make patient-level label table
# ---------------------------
def patient_level_label(group):
    """
    Decide one label per patient for stratification.
    
    Rule:
    - if patient has any malignant sample -> malignant (1)
    - else benign (0)
    """
    if (group["target_class_id"] == 1).any():
        return 1
    return 0
patient_df = (
    df.groupby("patient_id")
      .apply(patient_level_label)
      .reset_index(name="patient_label")
)

print("Unique patients:", len(patient_df))
print(patient_df["patient_label"].value_counts())

Unique patients: 912
patient_label
0    748
1    164
Name: count, dtype: int64


C:\Users\tahmi\AppData\Local\Temp\ipykernel_29548\3474198617.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(patient_level_label)


In [14]:
df.columns

Index(['image_id', 'patient_id', 'slice_index', 'z_position', 'sample_type',
       'target_class', 'target_class_id', 'malignancy_avg', 'class_id',
       'has_nodule', 'processing_status', 'error_message', 'dicom_path',
       'sop_instance_uid', 'image_path', 'mask_path',
       'largest_consensus_id_for_center', 'largest_consensus_area_resampled',
       'selected_non_nodule_index', 'negative_center_source',
       'crop_center_x_resampled', 'crop_center_y_resampled', 'crop_x1',
       'crop_y1', 'crop_x2', 'crop_y2', 'crop_size', 'original_spacing_x',
       'original_spacing_y', 'target_spacing_x', 'target_spacing_y',
       'original_height', 'original_width', 'resampled_height',
       'resampled_width', 'hu_min', 'hu_max', 'normalization',
       'num_consensus_nodules_original', 'num_consensus_nodules_in_crop',
       'malignancy_max'],
      dtype='object')

In [15]:
# ---------------------------
# Step 2: split patients
# Example: 70% train, 15% val, 15% test
# ---------------------------
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.35,
    random_state=44,
    stratify=patient_df["patient_label"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=44,
    stratify=temp_patients["patient_label"]
)


In [16]:
# ---------------------------
# Step 3: get patient ID sets
# ---------------------------
train_ids = set(train_patients["patient_id"])
val_ids = set(val_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

# safety check
print("Overlap train-val :", len(train_ids & val_ids))
print("Overlap train-test:", len(train_ids & test_ids))
print("Overlap val-test  :", len(val_ids & test_ids))

Overlap train-val : 0
Overlap train-test: 0
Overlap val-test  : 0


In [17]:
# ---------------------------
# Step 4: filter full dataframe
# ---------------------------
train_df = df[df["patient_id"].isin(train_ids)].copy()
val_df   = df[df["patient_id"].isin(val_ids)].copy()
test_df  = df[df["patient_id"].isin(test_ids)].copy()

print("Before balancing")
print("Train images:", len(train_df))
print("Val images  :", len(val_df))
print("Test images :", len(test_df))

print("\nTrain patient count:", train_df["patient_id"].nunique())
print("Val patient count  :", val_df["patient_id"].nunique())
print("Test patient count :", test_df["patient_id"].nunique())

print("\nTrain class counts before balancing:")
print(train_df["target_class"].value_counts())

print("\nVal class counts before balancing:")
print(val_df["target_class"].value_counts())

print("\nTest class counts before balancing:")
print(test_df["target_class"].value_counts())


# ---------------------------
# Step 5: balance each split by random pruning
# ---------------------------
def balance_split_by_pruning(split_df, class_col="target_class_id", random_state=42):
    """
    Randomly prune extra rows from the majority class so that
    all classes have the same number of samples.

    This keeps the subject-wise split unchanged because we do
    balancing only after patient-level splitting.
    """
    counts = split_df[class_col].value_counts()
    min_count = counts.min()

    balanced_parts = []
    for class_id, group in split_df.groupby(class_col):
        balanced_group = group.sample(n=min_count, random_state=random_state)
        balanced_parts.append(balanced_group)

    balanced_df = (
        pd.concat(balanced_parts, axis=0)
          .sample(frac=1, random_state=random_state)
          .reset_index(drop=True)
    )
    return balanced_df


train_df = balance_split_by_pruning(train_df, class_col="target_class_id", random_state=42)
val_df   = balance_split_by_pruning(val_df, class_col="target_class_id", random_state=42)
test_df  = balance_split_by_pruning(test_df, class_col="target_class_id", random_state=42)

print("\nAfter balancing")
print("Train images:", len(train_df))
print("Val images  :", len(val_df))
print("Test images :", len(test_df))

print("\nTrain class counts after balancing:")
print(train_df["target_class"].value_counts())

print("\nVal class counts after balancing:")
print(val_df["target_class"].value_counts())

print("\nTest class counts after balancing:")
print(test_df["target_class"].value_counts())

Before balancing
Train images: 1960
Val images  : 511
Test images : 556

Train patient count: 592
Val patient count  : 160
Test patient count : 160

Train class counts before balancing:
target_class
benign       1337
malignant     623
Name: count, dtype: int64

Val class counts before balancing:
target_class
benign       351
malignant    160
Name: count, dtype: int64

Test class counts before balancing:
target_class
benign       369
malignant    187
Name: count, dtype: int64

After balancing
Train images: 1246
Val images  : 320
Test images : 374

Train class counts after balancing:
target_class
malignant    623
benign       623
Name: count, dtype: int64

Val class counts after balancing:
target_class
malignant    160
benign       160
Name: count, dtype: int64

Test class counts after balancing:
target_class
malignant    187
benign       187
Name: count, dtype: int64


In [18]:
# ---------------------------
# Step 6: save balanced split CSVs
# ---------------------------
train_df.to_csv("train_split_balanced.csv", index=False)
val_df.to_csv("val_split_balanced.csv", index=False)
test_df.to_csv("test_split_balanced.csv", index=False)

print("\nSaved: train_split_balanced.csv, val_split_balanced.csv, test_split_balanced.csv")


Saved: train_split_balanced.csv, val_split_balanced.csv, test_split_balanced.csv


In [19]:
df_test = pd.read_csv("./test_split_balanced.csv")
print("test")
print(df_test["target_class"].value_counts())

print("----------------------------------")

df_train = pd.read_csv("./train_split_balanced.csv")
print("train")
print(df_train["target_class"].value_counts())

print("----------------------------------")

df_val = pd.read_csv("./val_split_balanced.csv")
print("validation")
print(df_val["target_class"].value_counts())

test
target_class
malignant    187
benign       187
Name: count, dtype: int64
----------------------------------
train
target_class
malignant    623
benign       623
Name: count, dtype: int64
----------------------------------
validation
target_class
malignant    160
benign       160
Name: count, dtype: int64
